# 07 - Unity Catalog Governance
Re-runnable. Runs after `04_gold_build` in the job so documentation and security are always reapplied.

| Area | What this notebook does |
|---|---|
| Documentation | Table + column comments on Gold |
| Discovery | Tags on catalog, schemas, tables; PII tags on columns |
| Privacy | Secure view masks customer identifiers unless user is in `pii_readers` |
| Access | Least privilege: analysts get Gold read-only via the secure view, no Bronze/Silver/raw PII |

In [0]:
%sql
ALTER CATALOG retail_lakehouse SET TAGS ('project' = 'retail_lakehouse', 'environment' = 'dev');

ALTER SCHEMA retail_lakehouse.bronze SET TAGS ('layer' = 'bronze', 'data_classification' = 'internal_raw');
ALTER SCHEMA retail_lakehouse.silver SET TAGS ('layer' = 'silver', 'data_classification' = 'internal_clean');
ALTER SCHEMA retail_lakehouse.gold   SET TAGS ('layer' = 'gold',   'data_classification' = 'business_ready');

COMMENT ON SCHEMA retail_lakehouse.bronze IS 'Raw data exactly as ingested by Auto Loader. Engineers only.';
COMMENT ON SCHEMA retail_lakehouse.silver IS 'Typed, validated, deduplicated data from the Lakeflow pipeline. Engineers only.';
COMMENT ON SCHEMA retail_lakehouse.gold   IS 'Star schema for analytics, dashboards and ML. Analysts read via secure views.';

In [0]:
%sql
ALTER TABLE retail_lakehouse.gold.fact_orders  SET TAGS ('domain' = 'sales',    'table_type' = 'fact',      'grain' = 'order_line_item');
ALTER TABLE retail_lakehouse.gold.dim_customer SET TAGS ('domain' = 'customer', 'table_type' = 'dimension', 'scd_type' = '2', 'contains_pii' = 'true');
ALTER TABLE retail_lakehouse.gold.dim_product  SET TAGS ('domain' = 'product',  'table_type' = 'dimension', 'scd_type' = '1');
ALTER TABLE retail_lakehouse.gold.dim_date     SET TAGS ('domain' = 'calendar', 'table_type' = 'dimension');

In [0]:
%sql
-- fact_orders
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN order_id          COMMENT 'Order identifier from source system';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN order_item_id     COMMENT 'Line number within the order (1, 2, 3...)';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN customer_sk       COMMENT 'FK to dim_customer. Point-in-time: customer version valid at purchase';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN product_sk        COMMENT 'FK to dim_product';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN purchase_date_key COMMENT 'FK to dim_date (yyyyMMdd)';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN price             COMMENT 'Item price in BRL';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN freight_value     COMMENT 'Shipping cost for this item in BRL';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN item_total        COMMENT 'price + freight_value in BRL';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN delivery_days     COMMENT 'Days from purchase to customer delivery. NULL if not delivered';
ALTER TABLE retail_lakehouse.gold.fact_orders ALTER COLUMN is_late_delivery  COMMENT 'TRUE if delivered after the estimated date. NULL if not delivered';

-- dim_customer
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_sk              COMMENT 'Surrogate key, one per customer version';
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_id              COMMENT 'Natural key from source system';
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_unique_id       COMMENT 'PII: identifies the real person across orders';
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_zip_code_prefix COMMENT 'PII (quasi-identifier): first 5 digits of zip code';
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN is_current               COMMENT 'TRUE for the latest version of this customer';

-- dim_product
ALTER TABLE retail_lakehouse.gold.dim_product ALTER COLUMN product_sk            COMMENT 'Surrogate key';
ALTER TABLE retail_lakehouse.gold.dim_product ALTER COLUMN product_category_name COMMENT 'Category in Portuguese; unknown if missing in source';
ALTER TABLE retail_lakehouse.gold.dim_product ALTER COLUMN product_volume_cm3    COMMENT 'length x height x width in cubic cm';

-- dim_date
ALTER TABLE retail_lakehouse.gold.dim_date ALTER COLUMN date_key   COMMENT 'Primary key, yyyyMMdd';
ALTER TABLE retail_lakehouse.gold.dim_date ALTER COLUMN is_weekend COMMENT 'TRUE for Saturday and Sunday';

In [0]:
%sql
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_unique_id       SET TAGS ('pii' = 'direct_identifier');
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_zip_code_prefix SET TAGS ('pii' = 'quasi_identifier');
ALTER TABLE retail_lakehouse.gold.dim_customer ALTER COLUMN customer_city            SET TAGS ('pii' = 'quasi_identifier');

In [0]:
%sql
CREATE OR REPLACE VIEW retail_lakehouse.gold.v_dim_customer_secure
COMMENT 'Analyst-facing customer dimension. PII masked unless the user is in the pii_readers group. Hashed IDs still support COUNT(DISTINCT).'
AS
SELECT
  customer_sk,
  customer_id,
  CASE WHEN is_account_group_member('pii_readers')
       THEN customer_unique_id
       ELSE sha2(customer_unique_id, 256)
  END AS customer_unique_id,
  CASE WHEN is_account_group_member('pii_readers')
       THEN customer_zip_code_prefix
       ELSE concat(left(customer_zip_code_prefix, 2), '***')
  END AS customer_zip_code_prefix,
  customer_city,
  customer_state,
  effective_from,
  effective_to,
  is_current
FROM retail_lakehouse.gold.dim_customer;

ALTER VIEW retail_lakehouse.gold.v_dim_customer_secure SET TAGS ('domain' = 'customer', 'access' = 'analyst', 'pii_masked' = 'true');

In [0]:
%sql
SELECT
  current_user()                          AS me,
  is_account_group_member('pii_readers')  AS i_can_see_pii;

SELECT 'raw table'   AS source, customer_unique_id, customer_zip_code_prefix, customer_city
FROM retail_lakehouse.gold.dim_customer WHERE customer_sk = 1
UNION ALL
SELECT 'secure view', customer_unique_id, customer_zip_code_prefix, customer_city
FROM retail_lakehouse.gold.v_dim_customer_secure WHERE customer_sk = 1;

In [0]:
%sql
-- Analysts can find the catalog and the Gold schema...
GRANT USE CATALOG ON CATALOG retail_lakehouse     TO `account users`;
GRANT USE SCHEMA  ON SCHEMA  retail_lakehouse.gold TO `account users`;

-- ...and read the fact, the safe dimensions, and the MASKED customer view
GRANT SELECT ON TABLE retail_lakehouse.gold.fact_orders           TO `account users`;
GRANT SELECT ON TABLE retail_lakehouse.gold.dim_product           TO `account users`;
GRANT SELECT ON TABLE retail_lakehouse.gold.dim_date              TO `account users`;
GRANT SELECT ON VIEW  retail_lakehouse.gold.v_dim_customer_secure TO `account users`;

-- Deliberately NOT granted: bronze, silver, gold.dim_customer (raw PII)

In [0]:
%sql
SHOW GRANTS `account users` ON SCHEMA retail_lakehouse.gold;

In [0]:
%sql
SHOW GRANTS ON TABLE retail_lakehouse.gold.dim_customer;

In [0]:
%sql
SELECT table_schema, table_name, table_type, comment
FROM retail_lakehouse.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'gold')
ORDER BY table_schema, table_name;

In [0]:
%sql
SELECT schema_name, table_name, column_name, tag_name, tag_value
FROM retail_lakehouse.information_schema.column_tags
WHERE tag_name = 'pii'
ORDER BY table_name, column_name;